# CodeCore AI — LoRA Fine-Tuning & GGUF Conversion for Android

Этот блокнот обучает LoRA-адаптер для модели **Qwen2.5-Coder-3B-Instruct** на вашем наборе данных из приложения CodeCore AI, а затем конвертирует и квантует веса в формат **GGUF Q4_K_M** для запуска непосредственно на смартфоне в `llama.cpp`.

In [ ]:
# 1. Установка необходимых зависимостей (Unsloth, TRL, bitsandbytes)
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# 2. Загрузка базовой модели Qwen2.5-Coder-3B в 4-битном режиме
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True
)

# Настройка LoRA-адаптера
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407
)

In [ ]:
# 3. Загрузка датасета dataset_android_codecore.jsonl из CodeCore AI
from google.colab import files
import json
from datasets import Dataset

print("Загрузите экспортированный из CodeCore AI файл: dataset_android_codecore.jsonl")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

data_rows = []
with open(filename, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            prompt = item['prompt']
            files_content = "\n".join([f"File:{file['path']}\n```kotlin\n{file['content']}\n```" for file in item['files']])
            
            text = f"<|im_start|>system\nТы — автономный senior Android-разработчик. Создай приложение на Kotlin и Jetpack Compose.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{files_content}<|im_end|>"
            data_rows.append({"text": text})

dataset = Dataset.from_list(data_rows)
print(f"Успешно загружено пар: {len(data_rows)}")

In [ ]:
# 4. Обучение LoRA модели
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit"
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 5. Сохранение и квантование в GGUF Q4_K_M для CodeCore AI
model.save_pretrained_gguf(
    "qwen2.5-coder-3b-codecore-trained",
    tokenizer,
    quantization_method = "q4_k_m"
)
print("Модель успешно конвертирована в GGUF! Скачайте полученный .gguf файл и импортируйте в приложение через SAF.")